# ISS Group 24 Modeling

**Before running:**
1. Open the [shared Drive folder](https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing) and click **"Add shortcut to Drive"** (anywhere in your Drive is fine).
2. Select a GPU runtime: *Runtime → Change runtime type → T4 GPU*.
3. Run cells **in order** top-to-bottom. Each stage cell is idempotent — re-running a completed stage skips it automatically via checkpoint detection.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


In [ ]:
SHARED_FOLDER_LINK = "https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing"
REPO_URL        = "https://github.com/pewpewnor/iss_group_24.git"
REPO_LOCAL_PATH = "/content/iss_group_24_src"


In [ ]:
from google.colab import drive


def mount_drive() -> str:
    """Mount Google Drive and return the iss_group_24 project root path."""
    drive.mount("/content/drive")
    for candidate in [
        "/content/drive/Shareddrives/iss_group_24",
        "/content/drive/MyDrive/iss_group_24",
    ]:
        if Path(candidate).exists():
            print(f"Drive mounted. Project root: {candidate}")
            return candidate
    raise RuntimeError(
        "Shared folder 'iss_group_24' not found on this Drive.\n"
        f"Open the link and add a shortcut to your Drive: {SHARED_FOLDER_LINK}"
    )


def setup_repo() -> None:
    """Clone the repo at depth=1, or reset to origin/main if already cloned."""
    if Path(REPO_LOCAL_PATH).exists():
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "fetch", "origin"], check=True)
        subprocess.run(
            ["git", "-C", REPO_LOCAL_PATH, "reset", "--hard", "origin/main"],
            check=True,
        )
        print(f"Repo updated to origin/main: {REPO_LOCAL_PATH}")
    else:
        subprocess.run(
            ["git", "clone", "--depth=1", REPO_URL, REPO_LOCAL_PATH],
            check=True,
        )
        print(f"Repo cloned: {REPO_LOCAL_PATH}")
    sys.path.insert(0, REPO_LOCAL_PATH)


def install_deps() -> None:
    """Install packages not pre-installed on Colab (torch/torchvision are pre-installed)."""
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pillow>=10.0", "scipy", "matplotlib>=3.7",
        ],
        check=True,
    )
    print("Dependencies ready")


In [ ]:
DRIVE_ROOT = mount_drive()
MANIFEST  = f"{DRIVE_ROOT}/dataset/cleaned/manifest.json"
DATA_ROOT = f"{DRIVE_ROOT}/dataset/cleaned"
MODEL_DIR = f"{DRIVE_ROOT}/model"

# Set CWD to the Drive project root so relative paths written by
# train.py (analysis/, manifest parent, etc.) land on the Drive.
os.chdir(DRIVE_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
setup_repo()

In [ ]:
install_deps()

In [ ]:
# requires setup_repo() to have run first
import aggregator
from modeling.train import train


---
## Step 0 — Build Dataset Manifest

Copies and indexes all images from `dataset/original/` into `dataset/cleaned/`.  
**Run once.** Skip if `manifest.json` already exists.

In [ ]:
if Path(MANIFEST).exists():
    print(f"Manifest already exists at {MANIFEST} — skipping aggregator.")
else:
    print("Running aggregator (this copies images to dataset/cleaned/ — may take several minutes)…")
    aggregator.main()

---
## Training

Shared kwargs used by every stage cell. Each stage cell overrides only the relevant epoch count; the resume mechanism automatically skips already-completed stages.

In [ ]:
TRAIN_KWARGS = dict(
    manifest         = MANIFEST,
    data_root        = DATA_ROOT,
    out_dir          = MODEL_DIR,
    episodes_per_epoch = 2000,
    val_episodes     = 400,
    batch_size       = 16,
    num_workers      = 2,   # Drive FUSE I/O is the bottleneck; 2 workers is sufficient
    resume           = True,  # auto-resume from model/last.pt; safe even without a checkpoint
    contrastive_weight_p1 = 0.15,  # nt_xent was still high at epoch 10 with default 0.1
    # all stage epochs off by default — each cell turns on exactly one
    phase1_frozen_epochs  = 0,
    phase1_partial_epochs = 0,
    phase2_frozen_epochs  = 0,
    phase2_partial_epochs = 0,
    phase2_full_epochs    = 0,
)


---
## Phase 1 — General-Domain Pretraining on VizWiz

**Goal:** Train the few-shot matcher on a large, diverse set of everyday-object categories before it has ever seen the target domain (HOTS / InsDet products).

**Dataset:** VizWiz-FewShot base split — 100 categories, ~4 229 images of objects photographed with a smartphone (blurry, poorly lit, varied angles). This domain gap from ImageNet makes the pretrained backbone features imperfect, which is exactly what we want to fix.

**What gets learned:** The three matcher heads — SupportTokenizer, CrossAttentionHead, and DetHead — learn to compare a query image against a one-shot support example and produce a bounding-box prediction. The backbone starts frozen so the heads learn the episode format first, then the upper backbone blocks are partially thawed to adapt feature representations to phone-photo appearance.

**Outputs:** `stage1_1.pt` (frozen heads warmup), `stage1_2.pt` (partial backbone finetune), and a rolling `best.pt` / `last.pt`.

### Stage 1.1 — VizWiz base · backbone frozen · heads LR 1e-3

Warms up the matcher heads (SupportTokenizer + CrossAttentionHead + DetHead) on 100 diverse VizWiz categories using the rotation-synthesis episode scheme. Backbone stays frozen — no ImageNet features are modified yet.

In [ ]:
train(**{**TRAIN_KWARGS, "phase1_frozen_epochs": 15})


### Stage 1.2 — VizWiz base · features[7:] @ 1e-5 · heads LR 5e-4

Partially unfreezes the upper MobileNetV3 blocks so the backbone can adapt its feature representations to the VizWiz scene domain while pretraining on diverse categories.

In [ ]:
train(**{**TRAIN_KWARGS, "phase1_partial_epochs": 30})


---
## Phase 2 — Target-Domain Transfer to HOTS + InsDet

**Goal:** Adapt the Phase 1 pretrained matcher to the actual evaluation domain: HOTS scene images and InsDet product photos, plus the 16 novel VizWiz document categories as additional support examples.

**Dataset:** HOTS scene RGBs + InsDet Background images + VizWiz query images form the negative pool; the 16 novel VizWiz support images (one per category, bbox annotated) are the positives. `attn_loss_weight` is raised to 1.0 because HOTS/InsDet annotations have reliable bbox targets.

**What gets learned:** The matcher specialises from broad everyday objects to the narrower product/document domain. Hard-negative mining (ratio 0.5) is active so the model learns to reject visually similar distractors. The final full-unfreeze stage lets low-level backbone filters drift slightly toward the target appearance.

**Outputs:** `stage2_1.pt`, `stage2_2.pt`, `stage2_3.pt`, and updated rolling `best.pt` / `last.pt`.

### Stage 2.1 — HOTS + InsDet + VizWiz novel · backbone re-frozen · heads LR 5e-4

Domain transfer warmup. Backbone is re-frozen to stabilize features while the matcher heads adapt from VizWiz pretraining to the target product/scene domain. Val switches to the HOTS+InsDet split.

In [ ]:
train(**{**TRAIN_KWARGS, "phase2_frozen_epochs": 15})


### Stage 2.2 — target domain · features[7:] @ 1e-5 · heads LR 5e-4

Main fine-tuning stage. Hard-negative mining is on (ratio 0.5). Contrastive weight bumped to 0.2. attn_loss_weight=1.0 (HOTS/InsDet have meaningful bbox attention targets).

In [ ]:
train(**{**TRAIN_KWARGS, "phase2_partial_epochs": 50})


### Stage 2.3 — full unfreeze · all backbone @ 5e-6 · heads LR 1e-4

Fine-tunes the entire network end-to-end at very low LR. Lower backbone layers stay at 5e-6 to prevent catastrophic forgetting of low-level ImageNet features.

In [ ]:
train(**{**TRAIN_KWARGS, "phase2_full_epochs": 15})
